In [1]:
from sim import run_test

run_test(alpha=0.1, beta=0.01, num_docs=1000, sents_per_doc=10, iterations=10)

Iteration 0 completed.
Iteration 1 completed.
Iteration 2 completed.
Iteration 3 completed.
Iteration 4 completed.
Iteration 5 completed.
Iteration 6 completed.
Iteration 7 completed.
Iteration 8 completed.
Iteration 9 completed.

Evaluation Results:
Mean Segmentation F1: 1.0000
Topic Classification Accuracy: 0.0627


In [ ]:
import numpy as np
import random
from scipy.special import logsumexp

# ==========================================
# 1. 仿真数据生成器 (Simulator)
# ==========================================
class TopWordsTopicSimulator:
    def __init__(self, num_topics=2, vocab_size=50, alpha=1.1, beta=1.1, gamma=(2, 2)):
        self.K = num_topics
        self.V = vocab_size
        self.alpha = np.full(self.K, alpha)
        self.beta = beta
        self.gamma = gamma
        self.chars = [chr(i + 0x4e00) for i in range(self.V)] 
        self.word_dict = self._generate_vocabulary()
        
        # 初始分布：主题-词分布 phi [cite: 228-233]
        self.phi = np.random.dirichlet([self.beta] * len(self.word_dict), size=self.K + 1)

    def _generate_vocabulary(self):
        vocab = []
        for _ in range(10): vocab.append("".join(random.sample(self.chars, 1)))
        for _ in range(15):
            base = random.sample(self.chars, 3)
            vocab.extend(["".join(base), "".join(base[:2]), "".join(base[1:])])
        return list(set(vocab))

    def generate_corpus(self, num_docs=10, sents_per_doc=10):
        corpus = []
        for d_id in range(num_docs):
            theta_d = np.random.dirichlet(self.alpha) 
            doc = {"doc_id": d_id, "theta_true": theta_d.tolist(), "sentences": []}
            for n_id in range(sents_per_doc):
                z_dn = np.random.choice(self.K, p=theta_d) + 1
                pi_dn = np.random.beta(self.gamma[0], self.gamma[1])
                
                words = []
                for _ in range(random.randint(4, 8)):
                    source = z_dn if np.random.rand() < pi_dn else 0
                    w_idx = np.random.choice(len(self.word_dict), p=self.phi[source])
                    words.append(self.word_dict[w_idx])
                
                doc["sentences"].append({
                    "text": "".join(words),
                    "gt_seg": words,
                    "gt_topic": int(z_dn),
                    "gt_pi": float(pi_dn)
                })
            corpus.append(doc)
        return corpus

# ==========================================
# 2. 模型核心 (EM & Inside-Outside)
# ==========================================

class TopWordsTopicModel:
    def __init__(self, dictionary, K, alpha=1.1, beta=1.1, gamma_prior=(2, 2)):
        self.W = dictionary
        self.V = len(dictionary)
        self.K = K
        self.alpha_p = alpha
        self.beta_p = beta
        self.gamma_p = gamma_prior
        self.phi = np.random.dirichlet([self.beta_p] * self.V, size=self.K + 1)
        self.word_map = {w: i for i, w in enumerate(self.W)}
        self.max_w_len = max(len(w) for w in self.W)

    def _inside_outside(self, text, phi_t, phi_0, pi_dn):
        """前向-后向算法计算期望计数 [cite: 315-318]"""
        L = len(text)
        # 混合权重 g_{k,pi}(w) [cite: 312-313]
        log_g = np.log(pi_dn * phi_t + (1 - pi_dn) * phi_0 + 1e-20)
        
        # Forward (Inside)
        f = np.full(L + 1, -np.inf); f[0] = 0.0
        for i in range(L):
            for l in range(1, self.max_w_len + 1):
                if i + l <= L:
                    w = text[i:i+l]
                    if w in self.word_map:
                        f[i+l] = np.logaddexp(f[i+l], f[i] + log_g[self.word_map[w]])
        
        if f[L] == -np.inf: return None, -np.inf
        
        # Backward (Outside)
        b = np.full(L + 1, -np.inf); b[L] = 0.0
        for i in range(L, 0, -1):
            for l in range(1, self.max_w_len + 1):
                if i - l >= 0:
                    w = text[i-l:i]
                    if w in self.word_map:
                        b[i-l] = np.logaddexp(b[i-l], b[i] + log_g[self.word_map[w]])
        
        # 计算期望词频 [cite: 317-318]
        counts = np.zeros(self.V)
        for i in range(L):
            for l in range(1, self.max_w_len + 1):
                if i + l <= L:
                    w = text[i:i+l]
                    if w in self.word_map:
                        idx = self.word_map[w]
                        prob = np.exp(f[i] + log_g[idx] + b[i+l] - f[L])
                        counts[idx] += prob
        return counts, f[L]

    def viterbi_segment(self, text, d_idx, n_idx, pi_val):
        """Viterbi 解码 [cite: 334]"""
        t_idx = np.argmax(self.theta[d_idx])
        log_g = np.log(pi_val * self.phi[t_idx+1] + (1 - pi_val) * self.phi[0] + 1e-20)
        L = len(text)
        dp = np.full(L + 1, -np.inf); dp[0] = 0.0
        ptr = [0] * (L + 1)
        for i in range(L):
            for l in range(1, self.max_w_len + 1):
                if i + l <= L:
                    w = text[i:i+l]
                    if w in self.word_map:
                        score = dp[i] + log_g[self.word_map[w]]
                        if score > dp[i+l]:
                            dp[i+l], ptr[i+l] = score, i
        res = []; curr = L
        while curr > 0:
            res.append(text[ptr[curr]:curr]); curr = ptr[curr]
        return res[::-1]

    def train(self, corpus, iterations=5):
        D = len(corpus)
        self.theta = np.random.dirichlet([self.alpha_p] * self.K, size=D)
        pi = [[0.5 for _ in d['sentences']] for d in corpus]
        
        for it in range(iterations):
            new_n_ti = np.zeros_like(self.phi)
            new_n_dt = np.zeros((D, self.K))
            c_spec_total = np.zeros((D, 100)) # 记录 pi 更新用的统计量
            c_bg_total = np.zeros((D, 100))
            
            for d in range(D):
                for n, sent in enumerate(corpus[d]['sentences']):
                    log_G = np.zeros(self.K)
                    e_counts_list = []
                    
                    # E-Step: 针对每个主题计算句子似然
                    for t in range(1, self.K + 1):
                        cnt, l_G = self._inside_outside(sent['text'], self.phi[t], self.phi[0], pi[d][n])
                        log_G[t-1] = l_G if cnt is not None else -1e10
                        e_counts_list.append(cnt)
                    
                    # 计算主题后验 gamma_dn(t) [cite: 404, 423]
                    log_post_z = log_G + np.log(self.theta[d] + 1e-20)
                    post_z = np.exp(log_post_z - logsumexp(log_post_z))
                    new_n_dt[d] += post_z
                    
                    # 累计词频统计量 
                    for t_idx, gamma_z in enumerate(post_z):
                        t_real = t_idx + 1
                        e_cnt = e_counts_list[t_idx]
                        if e_cnt is None: continue
                        
                        # 源后验 q_spec, q_bg [cite: 427]
                        denom = pi[d][n] * self.phi[t_real] + (1 - pi[d][n]) * self.phi[0] + 1e-20
                        q_s = (pi[d][n] * self.phi[t_real]) / denom
                        
                        new_n_ti[t_real] += gamma_z * q_s * e_cnt
                        new_n_ti[0] += gamma_z * (1 - q_s) * e_cnt
                        
                        c_spec_total[d, n] += gamma_z * np.sum(q_s * e_cnt)
                        c_bg_total[d, n] += gamma_z * np.sum((1 - q_s) * e_cnt)

            # M-Step 参数更新 [cite: 329-330, 414, 420]
            for t in range(self.K + 1):
                self.phi[t] = (new_n_ti[t] + self.beta_p - 1) / (np.sum(new_n_ti[t]) + self.V * (self.beta_p - 1) + 1e-20)
            for d in range(D):
                self.theta[d] = (new_n_dt[d] + self.alpha_p - 1) / (np.sum(new_n_dt[d]) + self.K * (self.alpha_p - 1) + 1e-20)
            
            # 更新 pi 
            for d in range(D):
                for n in range(len(pi[d])):
                    numer = (self.gamma_p[0] - 1) + c_spec_total[d, n]
                    denom = (sum(self.gamma_p) - 2) + c_spec_total[d, n] + c_bg_total[d, n]
                    pi[d][n] = numer / (denom + 1e-20)
                    
            print(f"Iteration {it} completed.")

# ==========================================
# 3. 运行与评价
# ==========================================
def run_test():
    # 生成仿真数据
    sim = TopWordsTopicSimulator(num_topics=2, alpha=0.1, beta=0.01)
    corpus = sim.generate_corpus(num_docs=1000, sents_per_doc=10)
    
    # 训练模型
    model = TopWordsTopicModel(sim.word_dict, K=2)
    model.train(corpus, iterations=10)
    
    # 评价指标
    seg_f1, topic_acc = [], []
    for d_idx, doc in enumerate(corpus):
        for n_idx, sent in enumerate(doc['sentences']):
            # 主题预测
            pred_t = np.argmax(model.theta[d_idx]) + 1
            topic_acc.append(1 if pred_t == sent['gt_topic'] else 0)
            
            # Viterbi 分词评价
            pred_seg = model.viterbi_segment(sent['text'], d_idx, n_idx, sent['gt_pi'])
            gt_set = set(sent['gt_seg']); pred_set = set(pred_seg)
            common = len(gt_set & pred_set)
            p = common / len(pred_set) if pred_set else 0
            r = common / len(gt_set) if gt_set else 0
            seg_f1.append(2*p*r/(p+r) if (p+r)>0 else 0)
            
    print(f"\nEvaluation Results:")
    print(f"Mean Segmentation F1: {np.mean(seg_f1):.4f}")
    print(f"Topic Classification Accuracy: {np.mean(topic_acc):.4f}")


run_test()

Iteration 0 completed.
Iteration 1 completed.
Iteration 2 completed.
Iteration 3 completed.
Iteration 4 completed.
Iteration 5 completed.
Iteration 6 completed.
Iteration 7 completed.
Iteration 8 completed.
Iteration 9 completed.

Evaluation Results:
Mean Segmentation F1: 1.0000
Topic Classification Accuracy: 0.0534
